# 🦙 量化教室：把 AI 塞進口袋的 4-bit 大冒險

> 一隻叫**咕咕夫**的駱馬教授、一個想回家的迷你語言模型**小史莫**、一個專吃記憶體的 **OOM 魔王** ——
> 以及你，量化學院的新生。

這本 Notebook 是一個**可以玩的教材**。你會一路學到：

| 章節 | 你會學到 | 你會做出 |
|:--|:--|:--|
| 序章　新生報到 | 怎麼玩、把環境準備好 | — |
| 第 1 章　模型為什麼這麼胖？ | 模型大小 = 參數數量 × 每個參數的 bit 數 | 模型體重計算機 |
| 第 2 章　手搖飲的秘密 | 量化 = 把數字「吸」到最近的格子 | `quantize` / `dequantize` |
| 第 3 章　一顆老鼠屎 | 離群值的災難、分塊量化 | 分塊量化器 |
| 第 4 章　Q4_0 解剖課 | llama.cpp 的 Q4_0 格式，一個 bit 都不放過 | **你自己的 Q4_0 量化器** |
| 第 5 章　GGUF 藏寶箱 | 下載真正的模型、轉成 GGUF、看懂檔案結構 | `smol-f16.gguf` |
| 第 6 章　親手量化 | 用你寫的量化器量化整個模型 | **`smol-Q4_0-handmade.gguf`** |
| 第 7 章　官方神器 | 業界怎麼做：`llama-quantize` | **`smol-Q4_0.gguf`** |
| 第 8 章　最終決戰 | 量化後還能用嗎？品質掉多少？ | 一張畢業證書 🎓 |

### 怎麼玩

1. **從上到下，一格一格執行**（點一下程式格，按 `Shift + Enter`）。不要跳著跑。
2. 看到 🎯 **任務**，就把程式碼裡的 `...` 換成你的答案，再執行。過關會拿到經驗值（XP），整章過關會拿到徽章。
3. 卡住了？執行 `quest.hint("任務編號")` 看提示。提示不扣分，多看幾次會越來越明確。
4. 進度會自動存在 `quest_save.json`。關掉再開，從第一格重新執行就能接著玩。

**需要準備**：Python 3.10 以上、可以上網、大約 2 GB 的硬碟空間。全程只用 CPU，不需要顯示卡。整趟旅程大約 2～3 小時。

---
## 序章　新生報到

### ▶ 啟動遊戲引擎

**每次打開這本 Notebook，都要先執行下面這一格。**

- 把 `"在這裡寫上你的名字"` 改成你的名字（它會印在畢業證書上）。
- 第一次執行時，如果電腦還沒裝需要的套件，它會自動幫你安裝（PyTorch 比較大，可能要等幾分鐘）。

In [ ]:
import importlib.util, os, platform, subprocess, sys

REPO = "https://github.com/Dot7dot/quatize-classroom.git"
if not os.path.isdir("quest"):                      # 只開了這一個 .ipynb（例如 Colab）→ 把整間教室抓下來
    if not os.path.isdir("quatize-classroom"):
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("quatize-classroom")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

NEED = ["numpy", "gguf", "huggingface_hub", "torch", "transformers", "matplotlib"]
missing = [m for m in NEED if importlib.util.find_spec(m) is None]
if missing:
    print("第一次啟動，正在安裝缺少的套件：", missing, "（PyTorch 比較大，請耐心等幾分鐘）")
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"]
    if platform.system() == "Linux":                # Linux 預設會抓好幾 GB 的顯卡版 PyTorch，這裡指定 CPU 版就好
        cmd += ["--extra-index-url", "https://download.pytorch.org/whl/cpu"]
    subprocess.run(cmd, check=True)

import numpy as np
import quest

quest.start("在這裡寫上你的名字")

### ▶ 故事開始

下面這一格會播放劇情。**點對話框**（或按空白鍵）看下一句。

In [ ]:
quest.scene("prologue")

### ▶ 裝備檢查

冒險之前先清點裝備。這一格會確認需要的 Python 套件都裝好了：

| 套件 | 用來做什麼 |
|:--|:--|
| `numpy` | 所有的數字運算（量化其實就是一堆陣列運算） |
| `gguf` | llama.cpp 官方的 Python 套件，用來讀寫 GGUF 檔 |
| `huggingface_hub` | 從 Hugging Face 下載模型 |
| `torch`、`transformers` | 官方轉檔腳本需要它們；最後也用來檢定量化後的品質 |
| `matplotlib` | 畫圖 |

In [ ]:
quest.checkpoint("0-2")

---
## 第 1 章　模型為什麼這麼胖？

In [ ]:
quest.scene("ch1")

### 1.1　模型就是一大堆數字

語言模型聽起來很神秘，但存在硬碟裡的時候，它就只是**一大堆數字**。這些數字叫做**權重（weights）**，也叫**參數（parameters）**。

小史莫的本名是 **SmolLM2-135M** —— 名字裡的 135M 就是「1 億 3 千 5 百萬個參數」的意思。精確地說是 **134,515,008** 個。

電腦用 **bit（位元）** 來存數字。存一個數字用的 bit 越多，就能存得越精確，但也越佔空間：

| 型別 | 用幾個 bit | 佔幾個 byte | 能表示多少種不同的值 |
|:--|--:|--:|--:|
| `float32`（單精度浮點數） | 32 | 4 | 約 43 億種 |
| `float16`（半精度浮點數） | 16 | 2 | 65,536 種 |
| 8-bit 整數 | 8 | 1 | 256 種 |
| **4-bit 整數** | **4** | **0.5** | **16 種** |

> 8 個 bit = 1 個 byte。

執行下面這格，親眼看看同一個數字 `0.1` 用不同的型別存起來長什麼樣子：

In [ ]:
x32 = np.float32(0.1)
x16 = np.float16(0.1)

print("float32：", x32.tobytes().hex(" "), f"（{x32.nbytes} bytes）  實際存到的值 =", f"{float(x32):.10f}")
print("float16：", x16.tobytes().hex(" "), f"      （{x16.nbytes} bytes）  實際存到的值 =", f"{float(x16):.10f}")
print()
print("float32 的 32 個 bit：", "".join(f"{b:08b}" for b in x32.tobytes()[::-1]))
print("float16 的 16 個 bit：", "".join(f"{b:08b}" for b in x16.tobytes()[::-1]))

注意到了嗎？**兩種型別都存不了「剛剛好的 0.1」**，只能存一個很接近的值。bit 越少，差得越多。

這個觀察很重要 —— 等一下我們要做的事情，本質上就是「故意用更少的 bit，接受一點點誤差」。

### 🎯 任務 1-1　幫小史莫量體重

寫一個函式，算出「有 `n_params` 個參數、每個參數用 `bits_per_weight` 個 bit」的模型有多大。

> 這門課統一用 **1 MB = 1,000,000 bytes**（跟檔案總管和 Hugging Face 網站顯示的一樣）。

把三個 `...` 換成你的算式，然後執行：

In [ ]:
def model_size_mb(n_params, bits_per_weight):
    """回傳模型大小（單位：MB）"""
    total_bits = ...         # 🖊️ 總共有幾個 bit？
    total_bytes = ...        # 🖊️ 8 個 bit 是 1 個 byte
    return ...               # 🖊️ 1,000,000 個 byte 是 1 MB

quest.check("1-1", model_size_mb)

用你剛寫好的函式，看看小史莫在不同精度下有多重：

In [ ]:
SMOL = 134_515_008      # 小史莫的參數數量

for name, bits in [("float32", 32), ("float16", 16), ("8-bit", 8), ("4-bit", 4)]:
    size = model_size_mb(SMOL, bits)
    print(f"{name:8s} {size:7.1f} MB  " + "█" * int(size / 12))

### 1.2　記憶體峽谷的磅秤

小史莫在 Hugging Face 上是用 16 bit 存的，所以是 269 MB。魔王的橋限重 100 MB。

打開磅秤玩玩看 —— 也可以順便看看 8B、70B 那種大模型有多誇張。

In [ ]:
quest.play("backpack")

### 🎯 任務 1-2　過橋需要幾 bit？

In [ ]:
quest.answer("1-2", ...)       # 🖊️ 把 ... 換成你在磅秤上找到的答案（一個整數）

### 📝 第 1 章重點

- 模型 = 一大堆數字（權重）。**模型大小 = 參數數量 × 每個參數的 bit 數 ÷ 8**。
- 參數的「數量」是訓練時就決定的，我們改不了；但「每個參數用幾個 bit」可以動手腳。
- 把 bit 數從 16 降到 4 左右，檔案就會剩下不到三分之一。這件事就叫做**量化（quantization）**。

問題是：16 bit 可以表示 65,536 種值，4 bit 只有 16 種。**只用 16 種值，要怎麼表示一個權重？** 下一章見分曉。

---
## 第 2 章　手搖飲的秘密

In [ ]:
quest.scene("ch2")

### 2.1　量化 ＝ 選最近的格子

假設飲料店規定糖只能加 0、1、2、3…… 「匙」，而一匙是 0.25 克。你想要 0.3 克的糖，店員會怎麼做？

1. 0.3 克 ÷ 0.25 = 1.2 匙
2. 沒有 1.2 匙這種東西，**四捨五入** → 1 匙
3. 實際拿到 1 × 0.25 = 0.25 克（跟你要的差了 0.05 克）

把「克」換成「權重」、「一匙的大小」換成 **scale**，這就是量化：

$$\text{代碼 } q = \text{round}\left(\frac{x}{\text{scale}}\right) \qquad\qquad \text{還原 } \hat{x} = q \times \text{scale}$$

- **x**：原本的權重（浮點數）
- **q**：量化後的**整數代碼** —— 這才是真正要存進檔案的東西
- **scale**：一格有多寬
- **x̂**：還原回來的值，會跟 x 差一點點。這個差距就是**量化誤差**

**4 個 bit 只能表示 16 個整數**。如果用有正負號的整數，範圍是 **−8 ～ 7**。
超出範圍的代碼只能硬塞進邊界 —— 這叫**截斷（clip）**。

```
代碼    -8  -7  -6  -5  -4  -3  -2  -1   0   1   2   3   4   5   6   7
         ┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼
                                      ▲ 每一格的寬度 = scale
```

### 🎯 任務 2-1　量化與還原

寫出 `quantize`（浮點數 → 代碼）和 `dequantize`（代碼 → 浮點數）。
會用到 `np.round`（四捨五入）和 `np.clip(陣列, 最小值, 最大值)`（夾進範圍）。

In [ ]:
def quantize(x, scale):
    """浮點數 → 4-bit 整數代碼（-8 ~ 7）"""
    q = ...                  # 🖊️ 除以 scale，再四捨五入
    q = ...                  # 🖊️ 把超出範圍的代碼夾回 -8 ~ 7
    return q.astype(np.int8)

def dequantize(q, scale):
    """整數代碼 → 浮點數"""
    return ...               # 🖊️

quest.check("2-1", quantize, dequantize)

拿幾個數字試試看。觀察「誤差」那一欄 —— 它永遠不會超過半格（scale ÷ 2）：

In [ ]:
x = np.array([0.03, -0.41, 0.77, 0.12, -0.95, 0.50], dtype=np.float32)
scale = 1 / 7

q = quantize(x, scale)
x_hat = dequantize(q, scale)

print(f"scale = {scale:.4f}（半格 = {scale / 2:.4f}）\n")
print("  原本的值   代碼   還原的值    誤差")
for a, b, c in zip(x, q, x_hat):
    print(f"  {a:8.4f}   {b:4d}   {c:8.4f}   {abs(a - c):.4f}")

### 2.2　甜度量化機

bit 數越多，格子越密，誤差越小。到底差多少？拉拉看就知道。

In [ ]:
quest.play("sugar")

### 🎯 任務 2-2　甜度量化機

In [ ]:
quest.answer("2-2", ...)       # 🖊️ 最少需要幾個 bit？

### 📝 第 2 章重點

- 量化三步驟：**除以 scale → 四捨五入 → 夾進範圍**。還原只要一步：**乘以 scale**。
- 存進檔案的是整數代碼。4 bit 的代碼只有 16 種可能（−8 ～ 7）。
- 每少 1 個 bit，格子數減半、誤差加倍。量化就是在「省空間」和「保精度」之間討價還價。

但還有一個關鍵問題沒回答：**scale 到底要設多少？** 設太小，大的權重會被截斷；設太大，格子太粗。下一章有人要來搗亂了。

---
## 第 3 章　一顆老鼠屎

In [ ]:
quest.scene("ch3")

### 3.1　scale 要怎麼選？

最直覺的做法：找出這批權重裡**絕對值最大**的那一個，讓它剛好對到最大的代碼 7。

$$\text{scale} = \frac{\max |x|}{7}$$

這樣所有權重都落在 −7 ～ +7 之間，不會有人被截斷。聽起來很完美？

來做個實驗：64 個正常的小權重，然後偷偷把其中一個換成超大的數字。

In [ ]:
rng = np.random.default_rng(0)
w_clean = (rng.standard_normal(64) * 0.1).astype(np.float32)     # 64 個正常的小權重
w_rat = w_clean.copy()
w_rat[45] = 2.0                                                    # 🐭 混進一顆老鼠屎

for name, w in [("沒有老鼠屎", w_clean), ("有老鼠屎　", w_rat)]:
    scale = np.max(np.abs(w)) / 7
    w_hat = dequantize(quantize(w, scale), scale)
    err = np.abs(w_hat - w).mean()
    zeros = int((w_hat == 0).sum())
    print(f"{name}  scale = {scale:.4f}   平均誤差 = {err:.4f}   被量化成 0 的權重：{zeros} / 64")

只多了**一個**大數字，scale 就被撐大十倍，結果一大半的權重直接被四捨五入成 0 —— 資訊全沒了。

真實的模型權重裡到處都有這種「大部分很小、偶爾幾個很大」的情況。怎麼辦？

### 3.2　老鼠屎實驗室

打開實驗室，把老鼠屎拉大，然後試試不同的「分塊」方式：

In [ ]:
quest.play("outlier")

### 🎯 任務 3-1　分塊量化

對策就是**分塊（block）**：把權重切成一小塊一小塊，**每一塊用自己的 scale**。老鼠屎只會害到它自己那一塊。

完成下面兩個函式。`blockwise_roundtrip` 會把每一塊「量化再還原」，讓我們看看誤差有多大
（裡面會用到你在第 2 章寫的 `quantize` 和 `dequantize`）。

In [ ]:
def absmax_scale(block):
    """算出這一塊的 scale：讓絕對值最大的權重剛好對到代碼 7"""
    return ...                                  # 🖊️

def blockwise_roundtrip(x, block_size):
    """每 block_size 個權重一塊，各自量化再還原"""
    out = np.empty_like(x)
    for start in range(0, len(x), block_size):
        block = x[start:start + block_size]
        scale = ...                             # 🖊️ 這一塊自己的 scale
        q = ...                                 # 🖊️ 量化這一塊
        out[start:start + block_size] = ...     # 🖊️ 還原這一塊
    return out

quest.check("3-1", absmax_scale, blockwise_roundtrip)

分塊不是免費的：**每一塊都要多存一個 scale**（不然解不回來）。如果 scale 用 16 bit 存，切得越細，scale 佔的比例就越高。

用你的函式跑跑看這筆帳：

In [ ]:
print("每塊幾個   平均誤差   每個權重平均花幾 bit")
for block_size in [64, 32, 16, 8, 4]:
    w_hat = blockwise_roundtrip(w_rat, block_size)
    err = np.abs(w_hat - w_rat).mean()
    bits = 4 + 16 / block_size                  # 4 bit 的代碼 + 分攤到的 scale
    print(f"   {block_size:3d}      {err:.4f}        {bits:.2f}")

### 🎯 任務 3-2　老鼠屎實驗

In [ ]:
quest.answer("3-2", ...)       # 🖊️ 每 4 個一塊的話，每個權重平均要幾個 bit？

### 📝 第 3 章重點

- scale 通常用「這一批權重的最大絕對值」來決定。
- 一個離群值就能毀掉一整批權重的精度。解法是**分塊量化**：每塊各自算 scale。
- 分塊越細越準，但要多存越多 scale。**每 32 個一塊**是個不錯的折衷：每個權重只多花 0.5 bit。

「4-bit 代碼 + 每 32 個權重一個 scale」—— 恭喜，你剛剛重新發明了 **Q4_0** 的骨架。下一章我們照著真正的規格把它做出來。

---
## 第 4 章　Q4_0 解剖課

In [ ]:
quest.scene("ch4")

### 4.1　Q4_0 的規格

**Q4_0** 是 llama.cpp 最經典的 4-bit 量化格式。規格非常精簡：

- 每 **32** 個權重為一個 **block**
- 每個 block 存成 **18 個 byte**：

```
┌──────────────┬─────────────────────────────────────────────────────┐
│  d (scale)   │              32 個 4-bit 代碼                         │
│  float16     │   兩個代碼擠一個 byte → 16 bytes                       │
│  2 bytes     │                                                     │
└──────────────┴─────────────────────────────────────────────────────┘
     2 bytes   +                 16 bytes                 = 18 bytes
```

18 bytes × 8 = 144 bit，分給 32 個權重 → **每個權重 4.5 bit**。原本用 float16 要 64 bytes，現在只要 18 bytes。

### 4.2　一格都不浪費的小聰明

第 2、3 章我們用代碼 −7 ～ +7，**−8 那一格完全沒用到**。16 格只用了 15 格，量化師看了會心痛。

Q4_0 的做法：

1. 找出這個 block 裡絕對值最大的權重 **m** —— 而且**保留它的正負號**。
2. 令 **d = m ÷ (−8)**。

這樣一來，m 自己的代碼就是 m ÷ d = **−8**，剛好站在最邊邊那一格上，誤差是 0。而且因為格子用滿了 8 格（不是 7 格），每一格比之前細了 12.5%。

> 有趣的副作用：如果 m 是正數，**d 就會是負數**。scale 是負的！沒關係，負負得正，算回來還是對的。

為了存成 0 ～ 15 的無號整數，代碼會再加上 8：

$$\text{代碼} = \text{clip}\big(\,\text{trunc}(x \div d + 8.5),\ 0,\ 15\big) \qquad\qquad \hat{x} = (\text{代碼} - 8) \times d$$

- `+ 8`：把 −8 ～ 7 平移成 0 ～ 15
- `+ 0.5` 再 `trunc`（去掉小數）：就是四捨五入
- `clip`：萬一算出 16（剛好在另一頭邊界），夾回 15

先到解剖台上看一次完整的過程：

In [ ]:
quest.play("dissect")

### 🎯 任務 4-1　Q4_0 編碼

輸入一個 block（32 個 float32），回傳 `d` 和 32 個代碼。

- 找「絕對值最大的元素」的位置：`np.argmax(np.abs(block))`
- 去掉小數：`np.trunc`

In [ ]:
def q4_0_encode(block):
    """32 個權重 → (d, codes)"""
    m = ...                  # 🖊️ 絕對值最大的那個權重（要保留正負號）
    d = ...                  # 🖊️ scale
    codes = ...              # 🖊️ 每個權重 ÷ d，加 8.5，去掉小數
    codes = ...              # 🖊️ 夾進 0 ~ 15
    return d, codes.astype(np.uint8)

quest.check("4-1", q4_0_encode)

In [ ]:
rng = np.random.default_rng(1)
block = (rng.standard_normal(32) * 0.05).astype(np.float32)

d, codes = q4_0_encode(block)
print("d     =", d)
print("codes =", codes)

### 4.3　兩個代碼擠一個 byte

每個代碼只需要 4 bit，但電腦最小的儲存單位是 byte（8 bit）。所以要把**兩個代碼塞進同一個 byte**。

Q4_0 的配對方式：第 `j` 個代碼放**低 4 bit**，第 `j + 16` 個代碼放**高 4 bit**。

```
代碼 lo = 8 = 0b1000      代碼 hi = 6 = 0b0110

hi << 4    →  0b0110_0000        （往左移 4 個 bit，等於乘以 16）
lo         →  0b0000_1000
              ───────────  用 | （OR）合在一起
byte       =  0b0110_1000  = 0x68
```

十六進位剛好一個字代表 4 bit，所以 `0x68` 一眼就能看出「高位是 6、低位是 8」。

### 🎯 任務 4-2　打包成 18 bytes

- 把數字變成 float16 的 2 個 byte：`np.float16(d).tobytes()`
- 位元運算：`|` 是 OR，`<< 4` 是左移 4 位

In [ ]:
def q4_0_pack(d, codes):
    """(d, 32 個代碼) → 18 bytes"""
    d_bytes = ...            # 🖊️ d 存成 float16（2 bytes）
    lo = ...                 # 🖊️ 前 16 個代碼
    hi = ...                 # 🖊️ 後 16 個代碼
    packed = ...             # 🖊️ 把 lo 和 hi 擠在一起（16 個 byte）
    return d_bytes + packed.astype(np.uint8).tobytes()

quest.check("4-2", q4_0_pack)

In [ ]:
raw = q4_0_pack(d, codes)
print(f"{len(raw)} bytes：", raw.hex(" ").upper())
print(f"原本用 float32 存要 {block.nbytes} bytes，用 float16 要 {block.nbytes // 2} bytes。")

### 🎯 任務 4-3　解碼還原

反過來做一次：從 18 bytes 解回 32 個權重。

- 從 bytes 讀出數字：`np.frombuffer(raw[:2], dtype=np.float16)[0]`
- 取出低 4 bit：`packed & 0x0F`　取出高 4 bit：`packed >> 4`

In [ ]:
def q4_0_decode(raw):
    """18 bytes → 32 個權重"""
    d = np.frombuffer(raw[:2], dtype=np.float16)[0]          # 前 2 bytes 是 d
    packed = np.frombuffer(raw[2:], dtype=np.uint8)          # 後 16 bytes 是代碼
    lo = ...                 # 🖊️ 每個 byte 的低 4 bit（前 16 個代碼）
    hi = ...                 # 🖊️ 每個 byte 的高 4 bit（後 16 個代碼）
    codes = np.concatenate([lo, hi])
    return ...               # 🖊️ (代碼 - 8) × d。提示：先用 codes.astype(np.float32) 轉成浮點數

quest.check("4-3", q4_0_decode)

In [ ]:
block_hat = q4_0_decode(raw)

print("   原本        還原       誤差")
for a, b in list(zip(block, block_hat))[:8]:
    print(f"{a:9.5f}  {b:9.5f}   {abs(a - b):.5f}")
print("   ……")
print(f"\n最大誤差 {np.abs(block - block_hat).max():.5f}，半格的寬度是 {abs(d) / 2:.5f}")

### 4.4　人肉量化機

換你當量化器。這是全課程唯一需要心算的地方。

In [ ]:
quest.play("human")

### 🎯 任務 4-4　人肉量化機

In [ ]:
quest.answer("4-4", "...")     # 🖊️ 把 ... 換成小遊戲給你的通關密語

### 4.5　向量化：一次處理幾百萬個 block

小史莫有 1.35 億個權重，也就是 **420 萬個 block**。如果用 Python 的 `for` 迴圈一個一個呼叫 `q4_0_encode`，要跑好幾分鐘。

numpy 的強項是**一次對整個陣列做運算**。把權重排成一個「每列 32 個」的表格，然後對**每一列**同時做一樣的事：

```
blocks.shape = (block 數, 32)         axis=1 的意思是「沿著每一列」

        ┌─ 32 個權重 ─┐
block 0 │ . . . . . . │ → m, d, 32 個代碼 → 18 bytes
block 1 │ . . . . . . │ → m, d, 32 個代碼 → 18 bytes
  ⋮
```

### 🎯 任務 4-5　向量化量化器

大部分已經寫好了，只剩兩個 `...` —— 邏輯跟任務 4-1 一模一樣，只是這次 `m`、`d` 都是「一整排」而不是一個數字。

這個函式就是第 6 章要拿來量化整個模型的武器。

In [ ]:
def q4_0_quantize(weights):
    """一次量化整個張量 → uint8 陣列，形狀 (block 數, 18)"""
    blocks = np.asarray(weights, dtype=np.float32).reshape(-1, 32)      # 每列一個 block
    idx = np.abs(blocks).argmax(axis=1, keepdims=True)                  # 每一列絕對值最大的位置
    m = np.take_along_axis(blocks, idx, axis=1)                         # 每個 block 的 m，形狀 (block 數, 1)
    d = ...                                                             # 🖊️ 跟 4-1 一樣。請用 np.float32(-8)
    with np.errstate(divide="ignore"):
        inv = np.where(d == 0, 0, 1 / d).astype(np.float32)             # 1 ÷ d（整塊都是 0 的話就當 0，避免除以零）
    codes = ...                                                         # 🖊️ blocks 乘以 inv，加 np.float32(8.5)，去掉小數
    codes = np.clip(codes, 0, 15).astype(np.uint8)
    packed = codes[:, :16] | (codes[:, 16:] << 4)                       # 兩個代碼擠一個 byte → (block 數, 16)
    d_bytes = d.astype(np.float16).view(np.uint8)                       # d 存成 float16 → (block 數, 2)
    return np.concatenate([d_bytes, packed], axis=1)                    # (block 數, 18)

quest.check("4-5", q4_0_quantize)

### 📝 第 4 章重點

- Q4_0：每 **32** 個權重一個 block，存成 **18 bytes**（2 bytes 的 scale `d` + 16 bytes 的代碼）→ **4.5 bit / 權重**。
- `d = m ÷ (−8)`，其中 m 是絕對值最大的權重（帶正負號）。這樣 16 個格子全部用上。
- 編碼：`trunc(x ÷ d + 8.5)` 夾進 0 ～ 15；解碼：`(代碼 − 8) × d`。
- 兩個 4-bit 代碼擠成一個 byte，4 個 bit 叫做一個 **nibble**。

你現在手上有一個**貨真價實、跟 llama.cpp 規格一致的 Q4_0 量化器**。理論課到此結束，接下來要對真正的模型下手了。

---
## 第 5 章　GGUF 藏寶箱

In [ ]:
quest.scene("ch5")

### 5.1　召喚小史莫

從 Hugging Face 把真正的模型下載回來。我們用的是
[**SmolLM2-135M-Instruct**](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct)
（Apache 2.0 授權，大約 270 MB）。檔案會放在 `workspace/models/` 資料夾。

In [ ]:
from quest import tools

MODEL_DIR = tools.download_model()

print("\n下載回來的檔案：")
for f in sorted(MODEL_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.stat().st_size / 1e6:8.2f} MB   {f.name}")

quest.checkpoint("5-1", model_dir=MODEL_DIR)

- `model.safetensors`：**權重本體**，幾乎所有的重量都在這裡。
- `config.json`：模型的設計圖（幾層、每層多寬……）。
- `tokenizer.json` 等：詞彙表，負責把文字切成模型看得懂的編號。

### 5.2　打開來看看

`.safetensors` 裡面裝的是一個個有名字的**張量（tensor）**—— 你可以把張量想成「一整塊排成表格的權重」。

In [ ]:
weights = tools.read_safetensors(MODEL_DIR / "model.safetensors")

print(f"總共 {len(weights)} 個張量。前幾個：\n")
for name in list(weights)[:11]:
    print(f"  {name:50s} 形狀 {str(weights[name].shape)}")

total = sum(w.size for w in weights.values())
print(f"\n全部加起來 {total:,} 個權重")

w = weights["model.layers.0.self_attn.q_proj.weight"]
print("\n第 0 層 q_proj 的前 6 個權重：", w[0, :6])

這些就是小史莫的「腦細胞」—— 一堆看起來沒什麼意義的小數。把 1.35 億個這種數字用對的方式乘在一起，就能寫出句子。

### 5.3　搬進 GGUF

**GGUF** 是 llama.cpp 使用的模型檔案格式。跟上面那一堆散落的檔案不同，GGUF **把權重、設計圖、詞彙表全部裝進同一個檔案**，
拿到任何支援 llama.cpp 的程式（Ollama、LM Studio、手機 App……）都能直接跑。

量化的標準流程分兩步：

```
Hugging Face 模型  ──①轉檔──▶  高精度的 GGUF（F16）  ──②量化──▶  Q4_0 的 GGUF
                convert_hf_to_gguf.py                llama-quantize
```

現在做第 ① 步。llama.cpp 官方提供了轉檔腳本 `convert_hf_to_gguf.py`，指令長這樣：

```
python convert_hf_to_gguf.py  <模型資料夾>  --outfile <輸出檔名>  --outtype f16
```

下面這格會先下載 llama.cpp 的原始碼（裡面有這支腳本），然後執行它。

> 萬一官方腳本因為網路或版本問題跑不起來，會自動改用咕咕夫教授寫的**備用轉檔器**
> （`quest/toolbox.py` 裡的 `backup_convert`，只認得小史莫，但產生的檔案內容一樣）。不會卡關，放心。

In [ ]:
import sys

F16_PATH = tools.F16_PATH

if tools.is_valid_gguf(F16_PATH):
    print(f"✔ {F16_PATH} 已經存在，不用重轉。（想重轉的話，先把這個檔案刪掉）")
else:
    convert_py = tools.get_convert_script()          # 下載官方轉檔腳本
    ok = False
    if convert_py:
        ok = quest.run(
            [sys.executable, convert_py, MODEL_DIR, "--outfile", F16_PATH, "--outtype", "f16"],
            title="Hugging Face → GGUF（F16）", log_name="convert",
        )
    if not ok:
        print("官方腳本這次沒有成功，改用教授的備用轉檔器 ……")
        tools.backup_convert(MODEL_DIR, F16_PATH)

quest.checkpoint("5-2", path=F16_PATH)

### 5.4　親手拆開檔頭

GGUF 檔案的**最前面 24 個 byte** 是檔頭，格式是固定的：

| 位置 | 內容 | 型別 | `struct` 的代號 |
|:--|:--|:--|:--|
| 第 0～3 byte | 魔法數字（magic） | 4 個字元 | `4s` |
| 第 4～7 byte | 版本 | 32-bit 無號整數 | `I` |
| 第 8～15 byte | 張量數量 | 64-bit 無號整數 | `Q` |
| 第 16～23 byte | 設定值（metadata）數量 | 64-bit 無號整數 | `Q` |

Python 內建的 `struct` 模組可以把一串 byte 照格式拆成數字。格式字串開頭加上 `<` 表示 **little-endian**（低位的 byte 寫在前面，一般電腦都是這樣）。

例如 `struct.unpack("<IH", raw)` 的意思是「little-endian，先一個 32-bit 整數，再一個 16-bit 整數」。

### 🎯 任務 5-3　親手讀檔頭

把 `"..."` 換成正確的格式字串：

In [ ]:
import struct

def read_gguf_header(path):
    with open(path, "rb") as f:
        raw = f.read(24)                                             # 檔頭固定是 24 個 byte
    magic, version, n_tensors, n_kv = struct.unpack("...", raw)      # 🖊️ 把 ... 換成格式字串
    return magic, version, n_tensors, n_kv

quest.check("5-3", read_gguf_header, F16_PATH)

### 5.5　用 `gguf` 套件看完整內容

檔頭之後的東西比較複雜，交給官方的 `gguf` 套件來讀：

In [ ]:
import gguf

reader = gguf.GGUFReader(F16_PATH)

print("══ 設定值（metadata）══")
for key, field in reader.fields.items():
    if not key.startswith("GGUF."):
        print(f"  {key:42s} {quest.kit.short(field.contents(), 56)}")

print("\n══ 張量（前 8 個）══")
for t in reader.tensors[:8]:
    shape = " × ".join(str(int(n)) for n in t.shape)
    print(f"  {t.name:26s} {shape:>12s}   {t.tensor_type.name:4s}  {t.n_bytes / 1e6:6.2f} MB")

看「張量」那一段的型別欄位：二維的大張量是 **F16**，一維的小張量（`norm`）是 **F32**。等一下我們要把 F16 那些變成 **Q4_0**。

最後，打開藏寶箱地圖，把整個檔案的構造看一遍。這張地圖畫的是**你自己的檔案**：

In [ ]:
quest.play("ggufmap", path=F16_PATH)

### 🎯 任務 5-4　藏寶箱地圖

In [ ]:
quest.answer("5-4", "...")     # 🖊️ 把 ... 換成那 4 個英文字母

### 📝 第 5 章重點

- 量化的標準流程：**Hugging Face 模型 → 轉成高精度 GGUF（F16）→ 量化成 Q4_0**。
- GGUF = 檔頭（24 bytes）+ 設定值 + 張量目錄 + 張量資料。**一個檔案就是一個完整的模型**。
- 檔案 99% 以上都是張量資料。每個張量在目錄裡都記著自己的**型別** —— 這就是量化要改的地方。

---
## 第 6 章　親手量化

In [ ]:
quest.scene("ch6")

### 6.1　先拿一個張量試刀

先確認要用哪一個量化器。如果你已經通過任務 4-5，接下來用的就是**你自己寫的** `q4_0_quantize`：

In [ ]:
import gguf
from quest import tools

F16_PATH, HANDMADE_PATH = tools.F16_PATH, tools.HANDMADE_PATH
reader = gguf.GGUFReader(F16_PATH)

quantizer = quest.my_or_prof("4-5", globals().get("q4_0_quantize"))

從模型裡挑一個張量 —— 第 0 層注意力機制的 `attn_q`，一個 576 × 576 的矩陣 —— 量化它，再解回來看看差多少。

In [ ]:
t = next(t for t in reader.tensors if t.name == "blk.0.attn_q.weight")
w = np.asarray(t.data, dtype=np.float32)                    # 原本的權重（F16 轉成 float32 方便計算）

raw = quantizer(w)                                          # ← 量化！
w_hat = quest.prof.q4_0_dequantize(raw).reshape(w.shape)    # 解回來，等一下比較用

print(f"張量 {t.name}，形狀 {w.shape}，共 {w.size:,} 個權重 = {w.size // 32:,} 個 block")
print(f"  F16  ：{t.n_bytes:>9,} bytes")
print(f"  Q4_0 ：{raw.nbytes:>9,} bytes（只剩 {raw.nbytes / t.n_bytes * 100:.1f}%）")
print()
print("原本：", np.round(w[0, :6], 4))
print("還原：", np.round(w_hat[0, :6], 4))

quest.checkpoint("6-1", original=w, restored=w_hat)

畫張圖更清楚。左圖是整個張量的分布（量化前後幾乎重疊）；右圖是第一個 block 的 32 個權重 —— 量化後的每個點都乖乖站在 16 條橫線的其中一條上。

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))

lim = float(np.percentile(np.abs(w), 99.5))                         # 只畫中間 99.5% 的範圍，比較看得清楚
ax1.hist(w.ravel(), bins=160, range=(-lim, lim), alpha=0.6, label="original (F16)")
ax1.hist(w_hat.ravel(), bins=160, range=(-lim, lim), alpha=0.6, label="after Q4_0")
ax1.set_title("All 331,776 weights of blk.0.attn_q")
ax1.set_xlabel("weight value")
ax1.legend()

first, first_hat = w.ravel()[:32], w_hat.ravel()[:32]
d = float(np.frombuffer(raw[0, :2].tobytes(), dtype=np.float16)[0])
for c in range(16):
    ax2.axhline((c - 8) * d, color="gray", lw=0.5, alpha=0.5)       # 16 levels
ax2.plot(first, "o", label="original")
ax2.plot(first_hat, "x", ms=9, mew=2, label="after Q4_0")
ax2.set_title("First block: 32 weights snapped onto 16 levels")
ax2.set_xlabel("weight index")
ax2.legend()
plt.tight_layout()
plt.show()

### 6.2　量化整個模型，寫成新的 GGUF

現在對全部 272 個張量做一樣的事。規則跟官方工具一樣：

- **二維的大張量** → 量化成 Q4_0（它們佔了 99.9% 的空間）
- **一維的小張量**（`norm`）→ 原封不動（量化它們省不了幾個 byte，卻很傷品質）

寫 GGUF 檔用的是 `gguf.GGUFWriter`，流程是：**抄設定值 → 加入每個張量 → 寫檔**。

下面這格就是整個量化流程的全貌。**仔細讀一遍，每一行你都看得懂了。**

In [ ]:
from gguf import GGMLQuantizationType, LlamaFileType

reader = gguf.GGUFReader(F16_PATH)                                       # 來源：F16 的 GGUF
arch = reader.fields["general.architecture"].contents()                   # 模型架構（"llama"）
writer = gguf.GGUFWriter(HANDMADE_PATH, arch)                             # 目的地：新的 GGUF

# ① 把設定值原封不動抄過去，只把「檔案型別」改成 MOSTLY_Q4_0
quest.kit.copy_metadata(reader, writer, file_type=LlamaFileType.MOSTLY_Q4_0)

# ② 一個一個處理張量
n_quantized = 0
for i, t in enumerate(reader.tensors):
    if quest.kit.should_quantize(t):                                      # 二維的大張量 → 量化
        w = np.asarray(t.data, dtype=np.float32)
        raw = quantizer(w)                                                # (block 數, 18)
        raw = raw.reshape(w.shape[0], -1)                                 # 排回「每列一排 block」的形狀
        writer.add_tensor(t.name, raw, raw_dtype=GGMLQuantizationType.Q4_0)
        n_quantized += 1
    else:                                                                 # 一維的小張量 → 照抄
        writer.add_tensor(t.name, np.asarray(t.data))
    if i % 40 == 0:
        print(f"  [{i + 1:3d}/{len(reader.tensors)}] {t.name}")

# ③ 寫進硬碟
writer.write_header_to_file()
writer.write_kv_data_to_file()
writer.write_tensors_to_file()
writer.close()

print(f"\n完成！量化了 {n_quantized} 個張量，其餘 {len(reader.tensors) - n_quantized} 個照抄。")

做出來了嗎？放上磅秤，看看過不過得了魔王的橋：

In [ ]:
quest.checkpoint("6-2", path=HANDMADE_PATH)

### 📝 第 6 章重點

- 量化一個模型 = **讀出每個張量 → 用量化器壓成 block → 連同設定值寫成新的 GGUF**。就這樣。
- 只量化二維的大張量；一維的小張量保留原樣。
- 你剛剛做出來的 `workspace/gguf/smol-Q4_0-handmade.gguf` 是一個**真的可以拿去用的模型檔**，
  任何支援 GGUF 的程式都讀得懂。

---
## 第 7 章　官方神器

In [ ]:
quest.scene("ch7")

### 7.1　取得 llama.cpp

llama.cpp 是用 C++ 寫的，要先有編譯好的執行檔才能用。下面這格會自動幫你準備：

1. 如果電腦上已經裝了 llama.cpp → 直接用
2. 否則從 GitHub 下載官方預先編譯好的版本（約 15～25 MB），放進 `workspace/llama.cpp-bin/`

In [ ]:
from quest import tools

llama = tools.get_llama_tools()

if llama is None:
    quest.note("⚠️ 這台電腦暫時拿不到 llama.cpp", *tools.install_help(),
               "沒關係，下一格會自動改用備案，不會卡關。")

### 7.2　一行指令

官方的量化工具叫 **`llama-quantize`**，用法是：

```
llama-quantize  <輸入的 GGUF>  <輸出的 GGUF>  <量化型別>
```

你花了四章才做出來的事，它一行就做完了。執行下面這格，然後比較**你的手工版**和**官方版**：

In [ ]:
F16_PATH, HANDMADE_PATH, OFFICIAL_PATH = tools.F16_PATH, tools.HANDMADE_PATH, tools.OFFICIAL_PATH

done = False
if llama is not None:
    done = quest.run([llama.quantize, F16_PATH, OFFICIAL_PATH, "Q4_0"],
                     title="用官方工具量化成 Q4_0", log_name="llama-quantize", env=llama.env)

if done:
    result = quest.kit.compare(HANDMADE_PATH, OFFICIAL_PATH)
    quest.checkpoint("7-1", official=OFFICIAL_PATH, result=result)
else:
    # 備案：拿官方 gguf 套件裡的 Q4_0 參考實作，跟你的量化器逐 byte 比對
    quantizer = quest.my_or_prof("4-5", globals().get("q4_0_quantize"))
    match = quest.kit.compare_with_reference(F16_PATH, quantizer)
    quest.checkpoint("7-1", match_pct=match)

### 7.3　解讀結果

如果官方工具有順利執行，你應該會看到兩件事：

**① 型別相同的張量，byte 幾乎（甚至完全）一模一樣。**
你用 numpy 寫的十幾行程式，跟 llama.cpp 裡的 C++ 程式算出了同樣的結果。Q4_0 就是這麼單純。

**② 官方版比較大，因為有一個張量被「升級」了。**
`token_embd.weight`（詞嵌入表）在你的版本是 Q4_0，在官方版卻是 **Q8_0**（8-bit）。

為什麼？小史莫的詞嵌入表**同時也是輸出層** —— 每產生一個字都要用到它，而且它一個就佔了全部權重的五分之一。
這麼關鍵的張量如果量化得太粗，品質會掉很多。所以官方工具寧可多花 14 MB，也要把它存得精細一點。

> 這就是為什麼檔案型別叫 **MOSTLY**_Q4_0 ——「**大部分**是 Q4_0」。
> 真實世界的量化不是把所有張量一視同仁，而是**把 bit 花在刀口上**。
> 你以後會看到的 `Q4_K_M`、`Q5_K_M`，就是把這個想法發揮到極致的混合配方。

到底升級這一個張量差多少？下一章我們直接用數據比。

---
## 第 8 章　最終決戰

In [ ]:
quest.scene("ch8")

### 8.1　讓小史莫開口說話

問三個版本的小史莫同一個問題：量化前的 F16、你的手工 Q4_0、官方的 Q4_0。

你可以把 `question` 換成任何問題再執行一次（小史莫主要會說英文，用英文問效果最好）。

> 如果「引擎」那一欄是 llama.cpp，順便看一下「速度」：量化後的模型通常**還比較快**，因為每算一個字要搬動的記憶體變少了。

In [ ]:
from quest import tools

MODEL_DIR = tools.MODEL_DIR
FILES = {
    "F16（量化前）": tools.F16_PATH,
    "手工 Q4_0": tools.HANDMADE_PATH,
    "官方 Q4_0": tools.OFFICIAL_PATH,
}
if "llama" not in globals():
    llama = tools.get_llama_tools(verbose=False)

question = "What is the capital of France? Answer in one sentence."

answers = quest.talk(question, FILES, llama=llama)
quest.checkpoint("8-1", answers=answers)

### 8.2　智商檢定

「看起來還能回答」不夠科學。我們需要一個**數字**。

語言模型最常用的指標是**困惑度（perplexity）**：讓模型讀一段文章，看它對「下一個字是什麼」平均有多意外。

- 困惑度 **越低越好**。
- 直覺上，困惑度 8 的意思是：模型在每個位置上，大約像是在 8 個選項之間猶豫。

做法：把每個 GGUF 檔裡的權重**解量化之後裝進 PyTorch 模型**，讓它讀同一段關於駱馬的短文
（llama.cpp 執行量化模型時，數學上做的是同一件事）。

In [ ]:
model, tok = quest.load_torch_model(MODEL_DIR)       # 用 PyTorch 載入模型的「骨架」

ppl = {}
for label, path in FILES.items():
    if path.exists():
        quest.kit.load_into_torch(model, path)       # 把這個 GGUF 的權重（解量化後）裝進去
        ppl[label] = quest.kit.perplexity(model, tok)
        print(f"{label:12s} {tools.mb(path):6.1f} MB   困惑度 = {ppl[label]:.3f}")

quest.checkpoint("8-2", ppl=ppl)

### 8.3　這些數字在說什麼

誠實地說：**量化不是免費的**。困惑度變高了，代表小史莫確實變得稍微沒那麼精準。

但是請注意幾件事：

- **有代價，但還堪用**。檔案只剩不到三分之一；困惑度升高了一些，剛才的問題卻還是答得出來。
- **手工版 vs 官方版**（如果你有跑出官方版）：官方只多花 14 MB 把詞嵌入表升級成 Q8_0，困惑度就明顯比較低。
  這就是「把 bit 花在刀口上」的威力。
- **模型越小，對量化越敏感**。小史莫只有 1.35 億個參數，每一個都身兼多職；
  幾十億參數的大模型做 4-bit 量化，品質損失通常小到幾乎感覺不出來。這也是為什麼大家都愛跑量化過的大模型。

魔王，你還有什麼話說？

In [ ]:
quest.boss()

### 8.4　畢業考

最後一關。六題選擇題，全部答對就能拿到通關密語。

In [ ]:
quest.play("quiz")

### 🎯 任務 8-3　畢業考

In [ ]:
quest.answer("8-3", "...")     # 🖊️ 把 ... 換成畢業考的通關密語

In [ ]:
quest.scene("ending")

---
## 終章　畢業典禮

執行下面這格，領取你的畢業證書。（如果還有任務沒完成，會先拿到一張「修業證明」，上面會列出還缺哪些。）

In [ ]:
quest.certificate()

### 🎁 你帶走了什麼

你的成果都在 `workspace/gguf/` 資料夾裡：

| 檔案 | 這是什麼 |
|:--|:--|
| `smol-f16.gguf` | 量化前的模型（F16） |
| `smol-Q4_0-handmade.gguf` | **你用自己寫的量化器做出來的 Q4_0 模型** |
| `smol-Q4_0.gguf` | 用官方 `llama-quantize` 做出來的 Q4_0 模型 |

這些都是標準的 GGUF 檔。有裝 llama.cpp 的話，在終端機輸入下面的指令就能跟小史莫聊天：

```
llama-cli -m workspace/gguf/smol-Q4_0.gguf
```

### 🚀 延伸挑戰

想繼續修煉的話：

1. **換個口味**：把第 7 章指令裡的 `Q4_0` 換成 `Q8_0` 或 `Q4_K_M`，比較檔案大小和困惑度。
   （劇透：`Q4_K_M` 的小史莫會超過 100 MB，過不了橋。想想看為什麼？）
2. **學官方那一招**：修改第 6 章的迴圈，遇到 `token_embd.weight` 時改用
   `gguf.quants.quantize(w, GGMLQuantizationType.Q8_0)`，型別也改成 `Q8_0`。困惑度進步多少？檔案胖了多少？
3. **自己寫 Q8_0**：規格更簡單 —— 每 32 個權重一個 float16 的 scale，每個權重用 8 bit（−127 ～ 127）。
4. **換一個模型**：試試大一點的 [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)（會說中文）。
   用官方的 `convert_hf_to_gguf.py` 和 `llama-quantize`，指令一模一樣。
5. **查查看**：什麼是 `imatrix`（importance matrix）？為什麼它能讓同樣 bit 數的量化品質更好？

### 📚 想知道更多

- llama.cpp：<https://github.com/ggml-org/llama.cpp>
- GGUF 格式說明：<https://github.com/ggml-org/ggml/blob/master/docs/gguf.md>
- 本課程使用的模型：[SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct)（Apache 2.0）

> 🦙「量化不是魔法，是四捨五入，加上一點點小聰明。」—— 咕咕夫教授